# QPSK coarse-prefix budget

This notebook slows down one narrow receive-side question: **how many symbols should the 4th-power front end average before the Costas loop takes over?**

The surprising answer in this repo version is that two things stop changing at different times:

- the **coarse frequency estimate** keeps getting cleaner as the prefix grows
- the **post-loop RMS** flattens much sooner once the handoff already lands inside pull-in range


## 1. Setup and scope boundary

We stay inside the bounded symbol-rate model from the repo. That means:

- QPSK symbols are already timed
- coarse acquisition uses the 4th-power estimate
- the carrier offset is fixed at `+0.62 rad/sample`, near the outer clean band but still inside the `\pi/4` alias limit
- only the coarse-prefix length and channel noise change

So this is not a general packet-design answer. It is a compact front-end budget card for this exact lab.


In [ ]:
from costaslab.analysis import study_coarse_prefix_budget

rows = study_coarse_prefix_budget(
    [8, 12, 16, 24, 32, 48, 64, 96, 128],
    [0.02, 0.04, 0.06, 0.08, 0.10],
    trials=12,
)
rows[:3]


## 2. The coarse front end keeps improving

The 4th-power estimate is averaging a noisy phase-difference statistic. More symbols should help, and they do.

What matters is **how fast** the error shrinks. A good practical checkpoint is 5 mrad/sample: below that, the residual drift over a few hundred symbols is already small enough that the loop is cleaning up a modest remainder instead of rescuing a sloppy front end.


In [ ]:
def checkpoint(noise):
    band = [row for row in rows if row.noise_std == noise]
    return [
        (row.coarse_prefix, round(row.mean_abs_coarse_frequency_error * 1000.0, 2))
        for row in band
    ]

checkpoint(0.08)


## 3. The loop output saturates earlier

The second half is the useful engineering point. Even when the coarse estimate keeps getting better, the tracked RMS may barely move.

That does **not** mean the extra prefix is fake. It means it is improving front-end honesty more than end-to-end visible output once the loop already has an easy cleanup job.


In [ ]:
noise = 0.08
band = [row for row in rows if row.noise_std == noise]
[(row.coarse_prefix, round(row.mean_tracked_rms_error, 5)) for row in band]


## 4. Read the artifact and push it further

The generated repo artifact is `assets/qpsk-coarse-prefix-budget.png`.

It pairs the two stories directly:

1. top panel: how the coarse estimate gets cleaner with more prefix
2. lower panel: how the loop output mostly plateaus after the handoff is already decent
3. side summary: where the 5 mrad checkpoint first gets crossed at each noise level

Good next questions:

- does this saturation point move if the residual offset is pushed closer to the alias edge?
- does pulse shaping or timing error make long prefixes more valuable?
- at what point does gain tuning matter less than front-end sample budget?

Useful companions:

- `reports/qpsk-coarse-prefix-budget.md`
- `reports/qpsk-frequency-acquisition.md`
- `assets/qpsk-acquisition-range-map.png`
